# Can we trust the numbers?

**Kalpa Retail, Week 1 Day 3.** Tuesday's finding reached the leadership group, and Anand Iyer,
the finance controller, replied to all:

> "Your dashboard says Q1 was Rs 2.1 crore. Our books say 1.9. Until your numbers match ours,
> Finance will not act on a drop measured from an ERP export. Send me a reconciliation."

The ERP team sent the raw exports with a note: the CSV was **stitched from two extracts during the
Q1 migration**.

Both figures are computable from data somebody has. Only one of them is right, and by the end of
this notebook you can say which and prove it in four lines.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

rows = kit.load_csv("C2_W01_D03_orders_STUDENT.csv")
print(f"{len(rows)} rows read from the export")
print(rows[0])

201 rows read from the export
{'order_id': 'KR-02001', 'customer_id': 'C-2000', 'segment': 'Retail-Core', 'channel': 'web', 'city': 'Mumbai', 'order_date': '2026-04-22', 'amount': '2200', 'status': 'delivered', 'quarter': 'Q1', 'discount': '50'}


## Everything read from a file is text

Monday's `TypeError` was one row. From a file it is every row, and it is silent until you try to
do arithmetic.

In [2]:
first = rows[0]
print(repr(first["amount"]), type(first["amount"]).__name__)
print(repr(first["order_date"]), type(first["order_date"]).__name__)

'2200' str
'2026-04-22' str


## The JSON feed does not open at all

In [3]:
import json

try:
    data = kit.load_json("C2_W01_D03_orders_STUDENT.json")
    print("parsed", len(data), "records")
except json.JSONDecodeError as e:
    print("json.decoder.JSONDecodeError:", e)

json.decoder.JSONDecodeError: Unterminated string starting at: line 1397 column 15 (char 27679)


### Reading a parse error

`Unterminated string` means a quote opened and never closed. The line, column and character offset
say exactly where the parser gave up. Open the file at that point before forming a theory.

In [4]:
text = kit.read_text("C2_W01_D03_orders_STUDENT.json")
print("the file ends like this:")
print(repr(text[-60:]))

the file ends like this:
'uarter": "Q2",\n  "discount": 150\n },\n {\n  "order_id": "KR-02'


The transfer was cut, so the file stops mid-record. **A truncated file is not a corrupt file.** It
is a complete file that ends early, and the fix is to ask for it again rather than to patch it.
The CSV carries the same orders, so today's work continues on that.

## MAP: profile before you analyse

Three counts per field. Any field where they disagree with what you expected is a finding.

In [5]:
kit.flow(["present", "convertible", "distinct"], lit=2,
         title="The three counts, and distinct is the one that catches a migration")

In [6]:
def profile(records, field):
    present = [r for r in records if r.get(field) not in (None, "")]
    convertible = 0
    for r in present:
        try:
            int(r[field])
            convertible += 1
        except (TypeError, ValueError):
            pass
    return present, convertible, len({r[field] for r in present})


report = []
for field in ("order_id", "customer_id", "segment", "amount", "status", "quarter"):
    present, convertible, distinct = profile(rows, field)
    report.append((field, len(present), convertible if field == "amount" else "n/a", distinct))
kit.table(["field", "present", "convertible", "distinct"], report,
          caption="The profile, field by field, on 201 rows")

field,present,convertible,distinct
order_id,201,n/a,186
customer_id,201,n/a,69
segment,201,n/a,4
amount,201,200,161
status,200,n/a,3
quarter,201,n/a,2


Three anomalies in one table.

1. **201 rows carry 186 distinct order ids.** Nobody mentioned that in the note.
2. **One amount will not convert.**
3. **One record has no status.**

In [7]:
ids = [r["order_id"] for r in rows]
repeated = {i for i in ids if ids.count(i) > 1}
bad_amount = [r for r in rows if not r["amount"].isdigit()]
no_status = [r for r in rows if not r["status"]]

print(f"order ids appearing more than once: {len(repeated)}")
print(f"amounts that will not convert:      {[(r['order_id'], r['amount']) for r in bad_amount]}")
print(f"records with no status:             {[r['order_id'] for r in no_status]}")

order ids appearing more than once: 15
amounts that will not convert:      [('KR-02063', 'twelve')]
records with no status:             ['KR-02119']


## Is the repetition the whole Rs 20 lakh?

Fifteen rows out of two hundred is seven percent and the gap is nine percent of the quarter. Close
is not the same as equal, and the arithmetic is the check.

In [8]:
def amount(r, default=None):
    try:
        return int(r["amount"])
    except (TypeError, ValueError):
        return default


seen, extra = set(), []
for r in rows:
    if r["order_id"] in seen:
        extra.append(r)
    seen.add(r["order_id"])

carried = sum(amount(r) or 0 for r in extra)
print(f"{len(extra)} repeated rows carry Rs {carried:,}")
print(f"the gap Anand named:      Rs {20_00_000:,}")

15 repeated rows carry Rs 2,003,710
the gap Anand named:      Rs 2,000,000


## The identity rule

A duplicate is not a row that looks the same. It is a row that **is the same order**, and you have
to say which fields decide that before removing anything.

In [9]:
import collections

by_id = collections.defaultdict(list)
for r in rows:
    by_id[r["order_id"]].append(r)

whole_record = sum(1 for v in by_id.values() if len(v) > 1
                   and len({tuple(sorted(x.items())) for x in v}) == 1)
differs = [k for k, v in by_id.items() if len(v) > 1
           and len({x["order_date"] for x in v}) > 1]
print(f"ids repeated with every field identical: {whole_record}")
print(f"ids repeated with a differing order_date: {differs}")

ids repeated with every field identical: 13
ids repeated with a differing order_date: ['KR-02151']


One pair shares an order id and differs on its date. **A whole-record check leaves it in.** A check
on the id removes one of them, and somebody has to say which date is right. That is a judgment,
and it belongs in the log rather than in the code.

In [10]:
kit.tree(
    {"label": "two rows match",
     "branches": [
         ("every field", {"label": "a duplicate, remove one"}),
         ("id only", {"label": "same order, two versions, pick and record"}),
         ("all but the id", {"label": "two real orders, keep both"}),
     ]},
    taken=["id only"], title="The identity rule, and where the judgment lives")

## The clean pass, with every decision recorded

In [11]:
decisions, clean, rejected = [], [], []
seen = set()
for r in rows:
    if r["order_id"] in seen:
        rejected.append((r, "duplicate order_id, the migration re-ran a batch"))
        continue
    seen.add(r["order_id"])
    if not r["status"]:
        rejected.append((r, "no status, so the order cannot be classified"))
        continue
    if amount(r) is None:
        rejected.append((r, "amount will not convert, and guessing it invents revenue"))
        continue
    clean.append(r)

reasons = collections.Counter(reason for _, reason in rejected)
kit.table(["reason", "rows"], sorted(reasons.items()), caption="The rejects log")

reason,rows
"amount will not convert, and guessing it invents revenue",1
"duplicate order_id, the migration re-ran a batch",15
"no status, so the order cannot be classified",1


### The only equation of the day

In [12]:
print(f"input    {len(rows)}")
print(f"clean    {len(clean)}")
print(f"rejected {len(rejected)}")
print(f"{len(clean)} + {len(rejected)} = {len(clean) + len(rejected)}")

kit.check("input equals clean plus rejected", len(clean) + len(rejected) == len(rows),
          f"{len(clean)} + {len(rejected)} against {len(rows)}")
kit.check("every rejected row carries a reason", all(reason for _, reason in rejected),
          f"{len(rejected)} rows")

input    201
clean    184
rejected 17
184 + 17 = 201


## The revenue bridge

It has to land on Finance's number, or a step is missing and the missing step is the finding.

In [13]:
q1_raw = sum(amount(r) or 0 for r in rows if r["quarter"] == "Q1")
q1_clean = sum(amount(r) for r in clean if r["quarter"] == "Q1")
q2_clean = sum(amount(r) for r in clean if r["quarter"] == "Q2")

print(f"Q1 as exported:  Rs {q1_raw:>12,}")
print(f"Q1 reconciled:   Rs {q1_clean:>12,}")
print(f"the gap:         Rs {q1_raw - q1_clean:>12,}")

kit.vflow([f"Rs {q1_raw:,} as exported",
           f"less {len(extra)} duplicate order ids",
           "less 1 row with no status and 1 amount that will not convert",
           f"Rs {q1_clean:,} reconciled"],
          lit=3, title="The revenue bridge, landing on Finance's number")

Q1 as exported:  Rs   20,998,210
Q1 reconciled:   Rs   18,998,210
the gap:         Rs    2,000,000


In [14]:
kit.check("the reconciled Q1 rounds to Finance's 1.9 crore",
          round(q1_clean / 1_00_00_000, 1) == 1.9, f"Rs {q1_clean:,}")
kit.check("the dashboard's Q1 rounds to 2.1 crore",
          round(q1_raw / 1_00_00_000, 1) == 2.1, f"Rs {q1_raw:,}")
kit.check("the gap is the Rs 20 lakh Anand named",
          abs((q1_raw - q1_clean) - 20_00_000) < 5000, f"Rs {q1_raw - q1_clean:,}")

**Anand was right.** The dashboard was counting a migration twice.

## The one that must not be cleaned

In [15]:
largest = max(clean, key=lambda r: amount(r))
print(f"largest surviving order: {largest['order_id']}, {largest['segment']}, "
      f"Rs {amount(largest):,}")
kit.check("the corporate order survived the pass", amount(largest) > 20_00_000,
          f"Rs {amount(largest):,}")

largest surviving order: KR-02186, Business, Rs 2,945,460


It is an outlier and it is real. It survives, and the log records that it was looked at and kept.
Cleaning is not making the data look tidy; remove every uncomfortable row and you have a dataset
that agrees with you.

## Recompute Tuesday, and say what changed

In [16]:
def opc(rs, seg=None):
    x = [r for r in rs if seg is None or r["segment"] == seg]
    return len(x) / len({r["customer_id"] for r in x})


c1 = [r for r in clean if r["quarter"] == "Q1"]
c2 = [r for r in clean if r["quarter"] == "Q2"]
kit.table(["measure", "Tuesday, as exported", "Wednesday, reconciled"],
          [("Q1 revenue", f"Rs {q1_raw:,}", f"Rs {q1_clean:,}"),
           ("revenue change", "down 11.0 percent",
            f"down {abs(100 * (q2_clean / q1_clean - 1)):.1f} percent"),
           ("orders per customer", "down 24.6 percent",
            f"down {abs(100 * (opc(c2) / opc(c1) - 1)):.1f} percent"),
           ("Retail-Plus", "down 49.0 percent",
            f"down {abs(100 * (opc(c2, 'Retail-Plus') / opc(c1, 'Retail-Plus') - 1)):.1f} percent"),
           ("Retail-Core", "down 5.3 percent",
            f"down {abs(100 * (opc(c2, 'Retail-Core') / opc(c1, 'Retail-Core') - 1)):.1f} percent")],
          caption="What moved once the data was reconciled")

measure,"Tuesday, as exported","Wednesday, reconciled"
Q1 revenue,"Rs 20,998,210","Rs 18,998,210"
revenue change,down 11.0 percent,down 1.6 percent
orders per customer,down 24.6 percent,down 12.9 percent
Retail-Plus,down 49.0 percent,down 33.3 percent
Retail-Core,down 5.3 percent,down 2.5 percent


In [17]:
plus = 100 * (opc(c2, "Retail-Plus") / opc(c1, "Retail-Plus") - 1)
core = 100 * (opc(c2, "Retail-Core") / opc(c1, "Retail-Core") - 1)
kit.check("the finding survives the clean pass", plus < -25, f"{plus:+.1f} percent")
kit.check("it is smaller than Tuesday reported", plus > -49, f"{plus:+.1f} against -49.0")
kit.check("Retail-Plus still falls far harder than Retail-Core", plus < core - 20,
          f"{plus:+.1f} against {core:+.1f}")
kit.check_summary()

## The note to Finance

> "Your 1.9 is correct. The export carried 201 rows against 186 distinct orders, because the Q1
> migration re-ran a batch. Removing the repeats, one row with no status and one amount that will
> not convert gives Rs 1.90 crore against your books. The decisions log is attached, and the one
> outlier in the file is a real corporate order which I have kept.
>
> Tuesday's finding moves with it. The revenue drop was mostly the migration, so it falls from 11
> percent to 1.6. The Retail-Plus frequency problem survives at 33 percent against Retail-Core's
> 2.5, which is smaller than I reported on Tuesday and still the thing to act on."

---

**The reputational risk is never the wrong number.** It is the wrong number that stayed up after
you knew. Tomorrow asks whether a 33 percent gap on this many customers is real at all.